# AttenX All-in-One GPU Notebook

This notebook consolidates the relevant execution flow in one place:
1. Environment and GPU checks
2. DAMSM pretraining
3. Main GAN training
4. Text-to-image generation
5. Preview generated outputs

It uses the existing project modules under `code/` and the existing `train.py` logic.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from PIL import Image
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
COLAB_REPO_URL = os.getenv("ATTENX_REPO_URL", "")
COLAB_REPO_DIR = Path("/content/AttenX_recheck_20260423")


def ensure_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd() / "AttenX_recheck_20260423",
        COLAB_REPO_DIR,
    ]

    for candidate in candidates:
        if (candidate / "code").exists():
            return candidate.resolve()

    if IN_COLAB:
        if not COLAB_REPO_URL:
            raise FileNotFoundError(
                "Repo not found in Colab. Set ATTENX_REPO_URL and rerun this cell, "
                "or clone the repo manually and cd into it."
            )

        if not COLAB_REPO_DIR.exists():
            print(f"Cloning repo into {COLAB_REPO_DIR} ...")
            subprocess.run(["git", "clone", COLAB_REPO_URL, str(COLAB_REPO_DIR)], check=True)

        if (COLAB_REPO_DIR / "code").exists():
            return COLAB_REPO_DIR.resolve()

    listing = os.listdir(".")
    raise FileNotFoundError(
        f"Could not find project root containing 'code'. Current directory has: {listing}"
    )


REPO_ROOT = ensure_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
CONFIG = {
    # Pipeline toggles
    'run_damsm': True,
    'run_train': True,
    'run_generate': True,

    # Device
    'device': 'auto',  # auto | cpu | cuda | cuda:0

    # Data and artifacts
    'data_dir': './data',
    'checkpoint_dir': './checkpoints',
    'damsm_output_dir': './checkpoints/damsm',
    'wordtoix_path': './data/birds/captions.pickle',
    'generation_output_dir': './results/attenx_notebook',

    # DAMSM pretrain
    'damsm_epochs': 1,
    'damsm_batch_size': 2,
    'damsm_lr': 2e-4,
    'damsm_save_interval': 1,

    # Main training
    'train_epochs': 1,
    'train_batch_size': 2,
    'num_workers': 0,
    'use_amp': True,

    # Model dims
    'ngf': 64,
    'ndf': 64,
    'nef': 512,
    'nhidden': 256,
    'nembed': 256,
    'vocab_size': 10000,
    'nz': 100,

    # Generation
    'seed': 42,
    'seq_len': 18,
    'prompts': [
        'a bright yellow bird with black wings',
        'a small red bird with a short beak and long tail',
        'a large blue bird sitting on a tree branch'
    ]
}

CONFIG

In [ ]:
from code.generator import G_NET
from code.encoder import RNN_ENCODER

def resolve_device(device_arg):
    if device_arg == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device_arg.startswith('cuda') and not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA was requested but is not available in this Python environment. '
            'Install a CUDA-enabled PyTorch build or use --device cpu.'
        )
    return torch.device(device_arg)

def _load_state_dict_checked(model, state_dict, model_name, allow_partial_load=False):
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if (missing or unexpected) and not allow_partial_load:
        raise RuntimeError(
            f'{model_name} checkpoint mismatch. Missing keys: {len(missing)}, '
            f'unexpected keys: {len(unexpected)}. '
            'Set allow_partial_load=True only if you understand the mismatch.'
        )
    if missing or unexpected:
        print(f'Warning: {model_name} partial mismatch (missing={len(missing)}, unexpected={len(unexpected)}).')

def _load_wordtoix(wordtoix_path):
    if not wordtoix_path or not os.path.isfile(wordtoix_path):
        raise FileNotFoundError(f'Word-to-index source not found: {wordtoix_path}')

    with open(wordtoix_path, 'rb') as f:
        raw = pickle.load(f, encoding='latin1')

    if isinstance(raw, dict) and 'wordtoix' in raw and isinstance(raw['wordtoix'], dict):
        return raw['wordtoix']
    if isinstance(raw, (list, tuple)) and len(raw) >= 4 and isinstance(raw[3], dict):
        return raw[3]

    raise ValueError(f'Unsupported wordtoix format in {wordtoix_path}')

def tokenize_prompt(text, wordtoix, seq_len=18):
    words = re.findall(r"[a-zA-Z']+", text.lower())
    if not words:
        words = ['unk']

    unk_id = wordtoix.get('<unk>', wordtoix.get('unk', 0))
    tokens = [wordtoix.get(w, unk_id) for w in words][:seq_len]
    cap_len = max(1, len(tokens))

    if len(tokens) < seq_len:
        tokens.extend([0] * (seq_len - len(tokens)))

    return torch.LongTensor(tokens), torch.LongTensor([cap_len])

def generate_images(
    prompts, checkpoint_path, damsm_text_path, wordtoix_path,
    ngf=64, nef=512, nhidden=256, nembed=256, nz=100,
    output_dir='results/attenx_notebook', seed=42, device_str='auto', seq_len=18, allow_partial_load=False
):
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f'Generator checkpoint not found: {checkpoint_path}')
    if not damsm_text_path or not os.path.isfile(damsm_text_path):
        raise FileNotFoundError(f'DAMSM text checkpoint not found: {damsm_text_path}')

    wordtoix = _load_wordtoix(wordtoix_path)
    vocab_size = max(wordtoix.values()) + 1 if wordtoix else 10000

    device = resolve_device(device_str)
    print('Generation device:', device)

    word_dim = nhidden * 2
    netG = G_NET(ngf=ngf, nz=nz, nef=nef, nhidden=nhidden, word_dim=word_dim).to(device)
    text_encoder = RNN_ENCODER(n_words=vocab_size, nhidden=nhidden, nembed=nembed).to(device)

    state = torch.load(checkpoint_path, map_location=device)
    if not isinstance(state, dict) or 'netG' not in state:
        raise ValueError(f'Invalid generator checkpoint format: {checkpoint_path}')
    _load_state_dict_checked(netG, state['netG'], 'Generator', allow_partial_load)

    ts = torch.load(damsm_text_path, map_location=device)
    if isinstance(ts, dict) and 'text_encoder' in ts:
        text_state = ts['text_encoder']
    elif isinstance(ts, dict) and 'state_dict' in ts:
        text_state = ts['state_dict']
    else:
        text_state = ts
    _load_state_dict_checked(text_encoder, text_state, 'Text encoder', allow_partial_load)

    text_encoder.eval()
    netG.eval()

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    os.makedirs(output_dir, exist_ok=True)
    saved_paths = []

    with torch.no_grad():
        for i, prompt in enumerate(prompts):
            caption, cap_len = tokenize_prompt(prompt, wordtoix, seq_len=seq_len)
            caption = caption.unsqueeze(0).to(device)
            cap_len = cap_len.to(device)

            words_emb, sent_emb = text_encoder(caption, cap_len, None)
            noise = torch.randn(1, nz, 1, 1, device=device)
            _, _, img_256, _, _ = netG(noise, sent_emb, words_emb)

            img = img_256.squeeze(0).detach().cpu().clamp(-1, 1)
            img = ((img + 1.0) * 127.5).byte().numpy()
            img = np.transpose(img, (1, 2, 0))

            save_path = os.path.join(output_dir, f'attenx_output_{i + 1:03d}.png')
            Image.fromarray(img).save(save_path, 'PNG')
            saved_paths.append(save_path)
            print(f'Saved: {save_path}')

    return saved_paths

In [ ]:
from code.datasets import TextImageDataset
from code.encoder import RNN_ENCODER, CNN_ENCODER
from code.losses import words_loss, sent_loss

def collate_text_image(batch):
    images, captions, cap_lens, cls_ids, keys = zip(*batch)
    images = torch.stack(images, 0)
    cap_lens = torch.stack(cap_lens, 0).squeeze(-1)
    captions = pad_sequence(captions, batch_first=True, padding_value=0)
    return images, captions, cap_lens, cls_ids, keys

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

def pretrain_damsm_notebook(cfg):
    set_seed(cfg['seed'])
    device = resolve_device(cfg['device'])
    pin_memory = device.type == 'cuda'
    print('DAMSM device:', device)

    dataset = TextImageDataset(data_dir=cfg['data_dir'], split='train', image_size=256)
    loader = DataLoader(
        dataset,
        batch_size=cfg['damsm_batch_size'],
        shuffle=True,
        num_workers=cfg['num_workers'],
        pin_memory=pin_memory,
        drop_last=True,
        collate_fn=collate_text_image,
    )

    text_encoder = RNN_ENCODER(
        n_words=cfg['vocab_size'], nhidden=cfg['nhidden'], nembed=cfg['nembed']
    ).to(device)
    image_encoder = CNN_ENCODER(nef=cfg['nef']).to(device)

    optimizer = torch.optim.Adam(
        list(text_encoder.parameters()) + list(image_encoder.parameters()),
        lr=cfg['damsm_lr'],
        betas=(0.5, 0.999),
    )

    os.makedirs(cfg['damsm_output_dir'], exist_ok=True)

    for epoch in range(cfg['damsm_epochs']):
        text_encoder.train()
        image_encoder.train()
        epoch_loss = 0.0
        n_batches = 0

        for imgs, captions, cap_lens, _, _ in loader:
            imgs = imgs.to(device)
            captions = captions.to(device)
            cap_lens = cap_lens.to(device).squeeze(-1)
            batch_size = imgs.size(0)

            cap_lens, sort_idx = torch.sort(cap_lens, descending=True)
            captions = captions[sort_idx]
            imgs = imgs[sort_idx]

            optimizer.zero_grad()
            words_emb, sent_emb = text_encoder(captions, cap_lens, None)
            features, cnn_code = image_encoder(imgs)

            w_loss = words_loss(features, words_emb.transpose(1, 2), None, cap_lens, batch_size)
            s_loss = sent_loss(cnn_code, sent_emb, None, batch_size)
            total_loss = w_loss + s_loss

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(text_encoder.parameters()) + list(image_encoder.parameters()),
                max_norm=5.0,
            )
            optimizer.step()

            epoch_loss += total_loss.item()
            n_batches += 1

        avg_loss = epoch_loss / max(1, n_batches)
        print(f'DAMSM Epoch {epoch + 1}/{cfg["damsm_epochs"]} | Loss: {avg_loss:.4f}')

        state = {
            'epoch': epoch,
            'text_encoder': text_encoder.state_dict(),
            'image_encoder': image_encoder.state_dict(),
        }
        latest_path = os.path.join(cfg['damsm_output_dir'], 'damsm_latest.pth')
        epoch_path = os.path.join(cfg['damsm_output_dir'], f'damsm_epoch_{epoch + 1:04d}.pth')
        torch.save(state, latest_path)
        if (epoch + 1) % cfg['damsm_save_interval'] == 0:
            torch.save(state, epoch_path)

    return os.path.join(cfg['damsm_output_dir'], 'damsm_latest.pth')

In [ ]:
from train import train as run_main_train

def train_gan_notebook(cfg, damsm_path):
    args = Namespace(
        data_dir=cfg['data_dir'],
        image_size=256,
        num_workers=cfg['num_workers'],
        ngf=cfg['ngf'],
        ndf=cfg['ndf'],
        nef=cfg['nef'],
        nhidden=cfg['nhidden'],
        nembed=cfg['nembed'],
        vocab_size=cfg['vocab_size'],
        epochs=cfg['train_epochs'],
        batch_size=cfg['train_batch_size'],
        lr_g=1e-4,
        lr_d=4e-4,
        D_steps=1,
        gamma_damsm=5.0,
        lambda_kl=2.0,
        clip_grad=None,
        seed=cfg['seed'],
        device=cfg['device'],
        use_amp=cfg['use_amp'],
        lr_patience=5,
        lr_factor=0.5,
        validate_interval=0,
        patience_early_stop=10,
        min_delta=1e-4,
        checkpoint_dir=cfg['checkpoint_dir'],
        checkpoint_interval=1,
        log_dir='./logs/training_notebook',
        resume=False,
        damsm_text_path=damsm_path,
        damsm_image_path=damsm_path,
    )

    run_main_train(args)
    return os.path.join(cfg['checkpoint_dir'], 'checkpoint_latest.pth')

In [ ]:
damsm_ckpt = os.path.join(CONFIG['damsm_output_dir'], 'damsm_latest.pth')
gen_ckpt = os.path.join(CONFIG['checkpoint_dir'], 'checkpoint_latest.pth')

if CONFIG['run_damsm']:
    damsm_ckpt = pretrain_damsm_notebook(CONFIG)

if CONFIG['run_train']:
    if not os.path.isfile(damsm_ckpt):
        raise FileNotFoundError(f'Missing DAMSM checkpoint: {damsm_ckpt}')
    gen_ckpt = train_gan_notebook(CONFIG, damsm_ckpt)

generated = []
if CONFIG['run_generate']:
    if not os.path.isfile(gen_ckpt):
        raise FileNotFoundError(f'Missing generator checkpoint: {gen_ckpt}')
    generated = generate_images(
        prompts=CONFIG['prompts'],
        checkpoint_path=gen_ckpt,
        damsm_text_path=damsm_ckpt,
        wordtoix_path=CONFIG['wordtoix_path'],
        ngf=CONFIG['ngf'],
        nef=CONFIG['nef'],
        nhidden=CONFIG['nhidden'],
        nembed=CONFIG['nembed'],
        nz=CONFIG['nz'],
        output_dir=CONFIG['generation_output_dir'],
        seed=CONFIG['seed'],
        device_str=CONFIG['device'],
        seq_len=CONFIG['seq_len'],
    )

print('DAMSM checkpoint:', damsm_ckpt)
print('Generator checkpoint:', gen_ckpt)
print('Generated files:', generated)

In [ ]:
out_dir = Path(CONFIG['generation_output_dir'])
paths = sorted(out_dir.glob('*.png'))

if not paths:
    print('No generated images found in', out_dir)
else:
    n = len(paths)
    cols = min(3, n)
    rows = (n + cols - 1) // cols

    plt.figure(figsize=(5 * cols, 5 * rows))
    for i, p in enumerate(paths, 1):
        plt.subplot(rows, cols, i)
        plt.imshow(Image.open(p))
        plt.title(p.name)
        plt.axis('off')
    plt.tight_layout()
    plt.show()